In [1]:
!pip install spacy gradio pandas scikit-learn nltk deep-translator langdetect google-genai transformers accelerate -q
!python -m spacy download en_core_web_trf -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
from google.colab import drive
import pandas as pd
import re

drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/twcs.csv')

print("total row:", len(df))
df.head()

Mounted at /content/drive
total row: 2811774


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


In [3]:
#total no of null row
print(df.isnull().sum())



tweet_id                         0
author_id                        0
inbound                          0
created_at                       0
text                             0
response_tweet_id          1040629
in_response_to_tweet_id     794335
dtype: int64


In [4]:
import re


df = df[(df['inbound'] == True) & (df['in_response_to_tweet_id'].isna())].copy()
print("customer tweets:", len(df))


def handle(text):
    m = re.match(r"^\s*@(\w+)", str(text))
    if m:
        return m.group(1).lower()
    return ""

df['handle'] = df['text'].apply(handle)

print("top 25 handles:")
print(df['handle'].value_counts().head(25))

customer tweets: 787346
top 25 handles:
handle
                   313932
applesupport        30928
uber_support        14954
amazonhelp          14864
delta               13265
americanair         11994
115858              11744
spotifycares        10582
tesco               10427
british_airways     10328
askplaystation       9955
115850               9560
xboxsupport          9132
southwestair         9132
virgintrains         9130
115873               9036
comcastcares         8690
ask_spectrum         8013
gwrhelp              7336
safaricom_care       6968
115821               6923
115888               6554
hulu_support         6544
sainsburys           6001
115911               5953
Name: count, dtype: int64


In [5]:
handle_to_dept = {
    "applesupport": "Tech Support", "xboxsupport": "Tech Support",
    "askplaystation": "Tech Support", "spotifycares": "Tech Support",
    "hulu_support": "Tech Support", "microsofthelps": "Tech Support",
    "delta": "Travel", "americanair": "Travel", "british_airways": "Travel",
    "southwestair": "Travel", "uber_support": "Travel", "virginatlantic": "Travel",
    "amazonhelp": "Retail", "asktarget": "Retail", "tesco": "Retail",
    "sainsburys": "Retail", "asdaserviceteam": "Retail", "ihelpbestbuy": "Retail",
    "sprintcare": "Telecom", "verizonsupport": "Telecom", "tmobilehelp": "Telecom",
    "comcastcares": "Telecom", "ask_spectrum": "Telecom", "virginmedia": "Telecom",
    "chasesupport": "Billing", "ask_wellsfargo": "Billing", "bofa_help": "Billing",
    "askpaypal": "Billing", "askamex": "Billing", "askciti": "Billing",
}

df['department'] = df['handle'].map(handle_to_dept)
df = df.dropna(subset=['department'])

df = df.sample(n=min(150000, len(df)), random_state=42)
df = df[['text', 'inbound', 'department']]

print("distribution:")
print(df['department'].value_counts())
print("\ntotal rows:", len(df))
df.head()

distribution:
department
Tech Support    51717
Travel          45729
Retail          24496
Telecom         19605
Billing          8453
Name: count, dtype: int64

total rows: 150000


,text,inbound,department
1494881,@AskPlayStation Yo my psn got hacked and the d...,True,Tech Support
1923683,@AppleSupport The new apple update completely ...,True,Tech Support
923489,@Tesco received my dot com shopping today but ...,True,Retail
2248435,@AppleSupport my AirPods getting disconnected ...,True,Tech Support
998930,"@AskPayPal Hello, I would like to have any hel...",True,Billing


In [6]:

def clean(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'\^\w+', '', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(clean)
df = df[df['clean_text'].str.split().str.len() >= 5]

row = df[df['text'].str.contains('http', na=False)].iloc[0]
print("before:", row['text'])
print("after :", row['clean_text'])

before: @AppleSupport my AirPods getting disconnected after a minute! Let me know what to do https://t.co/OCGCaUwRYf
after : my airpods getting disconnected after a minute let me know what to do


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report


X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['department'], test_size=0.2, random_state=42,
    stratify=df['department']
)

# baseline naive bayes
nb = make_pipeline(
    TfidfVectorizer(stop_words='english', max_features=5000),
    MultinomialNB()
)
nb.fit(X_train, y_train)
print("nb accuracy:", round(accuracy_score(y_test, nb.predict(X_test)) * 100, 2), "%")

# logistic regression model
clf = make_pipeline(
    TfidfVectorizer(
        stop_words='english',
        ngram_range=(1, 2),
        max_features=25000,
        min_df=3,
        sublinear_tf=True
    ),
    LogisticRegression(max_iter=1000, C=2.0)
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("lr accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print(classification_report(y_test, y_pred))

nb accuracy: 82.84 %
lr accuracy: 86.63 %
              precision    recall  f1-score   support

     Billing       0.74      0.58      0.65      1588
      Retail       0.85      0.84      0.85      4770
Tech Support       0.87      0.92      0.89     10023
     Telecom       0.86      0.77      0.82      3717
      Travel       0.89      0.91      0.90      8915

    accuracy                           0.87     29013
   macro avg       0.84      0.80      0.82     29013
weighted avg       0.86      0.87      0.86     29013



In [8]:
import nltk
import spacy
from nltk.sentiment.vader import SentimentIntensityAnalyzer

nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()     # between -1 to +1
ner = spacy.load("en_core_web_trf")    # for ententis
print("done")

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


done


In [9]:
from textblob import TextBlob
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# manually labelled test complaints
sent_data = [
    ("Worst airline ever, my flight was cancelled and nobody helped!", "Very Negative"),
    ("My bank account was hacked and all my money is stolen, furious!", "Very Negative"),
    ("Terrible customer service, completely ruined my trip.", "Very Negative"),
    ("Horrible network speed, cannot even load a webpage.", "Very Negative"),
    ("My order from last Friday still has not arrived.", "Negative"),
    ("The app keeps crashing after the new update.", "Negative"),
    ("I was charged twice on my credit card statement.", "Negative"),
    ("Wifi disconnects every ten minutes at home.", "Negative"),
    ("Can you check the status of my flight tomorrow?", "Neutral"),
    ("I need to update my billing address on the account.", "Neutral"),
    ("What are your customer support hours on Sunday?", "Neutral"),
    ("Please share the tracking link for my package.", "Neutral"),
    ("Thank you for resolving my refund so quickly!", "Positive"),
    ("Great support from the technician today, issue is fixed.", "Positive"),
    ("Really happy with the fast delivery and smooth service.", "Positive"),
    ("Awesome help from your team, appreciate the quick reply!", "Positive"),
]

def vader_lbl(t):
    sc = sia.polarity_scores(t)['compound']
    if sc <= -0.5:
        return "Very Negative"
    elif sc < -0.05:
        return "Negative"
    elif sc > 0.05:
        return "Positive"
    return "Neutral"

def tb_lbl(t):
    pol = TextBlob(t).sentiment.polarity
    if pol <= -0.5:
        return "Very Negative"
    elif pol < -0.05:
        return "Negative"
    elif pol > 0.05:
        return "Positive"
    return "Neutral"

y_true = [y for _, y in sent_data]
y_vader = [vader_lbl(x) for x, _ in sent_data]
y_tb = [tb_lbl(x) for x, _ in sent_data]

lbls = ["Very Negative", "Negative", "Neutral", "Positive"]
print("vader accuracy:", round(accuracy_score(y_true, y_vader) * 100, 2), "%")
print("textblob accuracy:", round(accuracy_score(y_true, y_tb) * 100, 2), "%")
print("\nvader report:")
print(classification_report(y_true, y_vader, labels=lbls, zero_division=0))
print("confusion matrix (vader):")
print(confusion_matrix(y_true, y_vader, labels=lbls))

vader accuracy: 62.5 %
textblob accuracy: 62.5 %

vader report:
               precision    recall  f1-score   support

Very Negative       1.00      1.00      1.00         4
     Negative       0.00      0.00      0.00         4
      Neutral       0.40      0.50      0.44         4
     Positive       0.57      1.00      0.73         4

     accuracy                           0.62        16
    macro avg       0.49      0.62      0.54        16
 weighted avg       0.49      0.62      0.54        16

confusion matrix (vader):
[[4 0 0 0]
 [0 0 3 1]
 [0 0 2 2]
 [0 0 0 4]]


In [10]:
# manually annotated entity test set
ner_data = [
    ("My Amazon order never arrived in Pune last Friday.", {("Amazon", "ORG"), ("Pune", "GPE"), ("last Friday", "DATE")}),
    ("Delta Airlines cancelled my flight from Mumbai to London.", {("Delta Airlines", "ORG"), ("Mumbai", "GPE"), ("London", "GPE")}),
    ("Chase Bank charged me 50 dollars yesterday.", {("Chase Bank", "ORG"), ("50 dollars", "MONEY"), ("yesterday", "DATE")}),
    ("Apple Support said my iPhone warranty expired in January.", {("Apple Support", "ORG"), ("iPhone", "PRODUCT"), ("January", "DATE")}),
    ("Verizon internet has been down in New York since Monday.", {("Verizon", "ORG"), ("New York", "GPE"), ("Monday", "DATE")}),
    ("PayPal froze my account in Delhi on Tuesday.", {("PayPal", "ORG"), ("Delhi", "GPE"), ("Tuesday", "DATE")}),
    ("Uber driver overcharged me in Bangalore last night.", {("Uber", "ORG"), ("Bangalore", "GPE"), ("last night", "TIME")}),
    ("Spotify billed my Visa card twice this month.", {("Spotify", "ORG"), ("Visa", "ORG"), ("this month", "DATE")}),
]

tp = 0
fp = 0
fn = 0

for txt, gold in ner_data:
    doc = ner(txt)
    pred = {(e.text, e.label_) for e in doc.ents}
    tp += len(pred & gold)
    fp += len(pred - gold)
    fn += len(gold - pred)
    print("text:", txt)
    print("expected :", gold)
    print("predicted:", pred)
    print("-" * 30)

prec = tp / (tp + fp) if (tp + fp) > 0 else 0
rec = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0

print(f"ner precision: {round(prec * 100, 2)} %")
print(f"ner recall   : {round(rec * 100, 2)} %")
print(f"ner f1-score : {round(f1 * 100, 2)} %")

text: My Amazon order never arrived in Pune last Friday.
expected : {('last Friday', 'DATE'), ('Pune', 'GPE'), ('Amazon', 'ORG')}
predicted: {('last Friday', 'DATE'), ('Pune', 'GPE'), ('Amazon', 'ORG')}
------------------------------
text: Delta Airlines cancelled my flight from Mumbai to London.
expected : {('Mumbai', 'GPE'), ('London', 'GPE'), ('Delta Airlines', 'ORG')}
predicted: {('Mumbai', 'GPE'), ('London', 'GPE'), ('Delta Airlines', 'ORG')}
------------------------------
text: Chase Bank charged me 50 dollars yesterday.
expected : {('yesterday', 'DATE'), ('50 dollars', 'MONEY'), ('Chase Bank', 'ORG')}
predicted: {('yesterday', 'DATE'), ('50 dollars', 'MONEY'), ('Chase Bank', 'ORG')}
------------------------------
text: Apple Support said my iPhone warranty expired in January.
expected : {('January', 'DATE'), ('iPhone', 'PRODUCT'), ('Apple Support', 'ORG')}
predicted: {('iPhone', 'PRODUCT'), ('Apple Support', 'ORG'), ('January', 'DATE')}
------------------------------
text: Veriz

In [11]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

sample_tweets = df['clean_text'].sample(min(50000, len(df)), random_state=42)

lda_vec = CountVectorizer(max_df=0.9, min_df=10, stop_words='english', max_features=2000)
dtm = lda_vec.fit_transform(sample_tweets)

lda = LatentDirichletAllocation(n_components=5, learning_method='online', random_state=42)
lda.fit(dtm)

words = lda_vec.get_feature_names_out()
for i, topic in enumerate(lda.components_):
    top_indices = topic.argsort()[-8:][::-1]
    top_words = [words[idx] for idx in top_indices]
    print(f"Topic {i + 1}:", ", ".join(top_words))

Topic 1: internet, going, order, just, want, guys, days, like
Topic 2: account, help, dont, email, card, hi, does, working
Topic 3: flight, service, trying, im, customer, need, help, time
Topic 4: phone, app, iphone, help, update, fix, ios, new
Topic 5: having, store, im, issues, battery, received, 10, minutes


In [12]:
import torch
from langdetect import detect
from deep_translator import GoogleTranslator
from google import genai
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM

#gemini api
try:
    key = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=key)
    print("gemini ready")
except Exception as e:
    print("gemini error:", e)
    client = None

g_model = "gemini-3.5-flash-lite"

#qwen model
q_name = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(q_name)
qwen = AutoModelForCausalLM.from_pretrained(q_name, device_map="auto")
print("qwen loaded on:", qwen.device)

gemini ready


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

qwen loaded on: cuda:0


In [13]:
#this cell is about if the input is in other lang it will translate into english

def trans(txt):
    try:
        lang = detect(txt)
    except Exception:
        lang = "en"

    if lang != "en":
        try:
            eng = GoogleTranslator(source='auto', target='en').translate(txt)
            if eng and eng.strip():
                return lang, eng.strip(), True
        except Exception as e:
            if client is not None:
                try:
                    res = client.models.generate_content(
                        model=g_model,
                        contents=f"translate this complaint to english only:\n{txt}"
                    )
                    if res.text and res.text.strip():
                        return lang, res.text.strip(), True
                except Exception as e2:
                    print("trans error:", e2)
            else:
                print("trans error:", e)
        return lang, "", False

    return lang, txt, True


# internal note for support team (qwen)
def brief(txt, dept, prio, ents):
    p = f"""write a short plain-text note for the {dept} team. no markdown, stars, or hashtags.
complaint: {txt}
priority: {prio}
entities: {ents}

use this exact format:
summary: <1 short sentence about the customer issue>
next steps:
- <first check>
- <second check>"""

    try:
        msg = [
            {"role": "system", "content": "you write short plain-text support notes without markdown, stars, or hashtags."},
            {"role": "user", "content": p}
        ]
        chat = tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
        inp = tok([chat], return_tensors="pt").to(qwen.device)

        with torch.no_grad():
            out_ids = qwen.generate(
                **inp,
                max_new_tokens=95,
                temperature=0.2,
                top_p=0.9,
                do_sample=True,
                pad_token_id=tok.eos_token_id
            )

        res = tok.decode(out_ids[0][inp.input_ids.shape[-1]:], skip_special_tokens=True)
        res = re.sub(r'\*+', '', res)
        res = re.sub(r'#+\s*', '', res)
        return res.strip()
    except Exception as e:
        print("qwen error:", e)
        return f"summary: {dept} issue ({prio})"


# polite reply for customer (gemini)
def reply(txt, dept, sent, prio, ents):
    p = f"""write a polite 3 sentence customer support reply for the {dept} team.
mention key details, match urgency if priority is high, and do not make up order numbers or refunds.
complaint: {txt}
sentiment: {sent}
priority: {prio}
entities: {ents}"""

    if client is not None:
        try:
            res = client.models.generate_content(model=g_model, contents=p)
            return res.text.strip()
        except Exception as e:
            print("gemini error:", e)

    return f"sorry for the trouble. your issue is sent to {dept} with {prio} and our team will contact you soon."

In [14]:
# main pipeline
def proc(txt):
    if txt.strip() == "":
        return None, "", "", "", "", "", "", ""

    lang, eng, ok = trans(txt)
    if not ok or eng.strip() == "":
        return None, "error", "error", "none", lang, "translation failed", "error: translation failed", "unable to translate complaint. please try again or enter in english."

    eng_show = "not needed" if lang == "en" else eng
    cln = clean(eng)
    if not re.search(r'[a-z]', cln):
        return None, "error", "error", "none", lang, eng_show, "error: no valid words found", "please enter a valid complaint with words."

    # department probabilities
    probs = clf.predict_proba([cln])[0]
    scores = {c: float(p) for c, p in zip(clf.classes_, probs)}
    dept = clf.predict([cln])[0]

    # sentiment and priority
    sc = sia.polarity_scores(eng)['compound']
    if sc <= -0.5:
        sent = "Very Negative"
        prio = "High Priority"
    elif sc < -0.05:
        sent = "Negative"
        prio = "Medium Priority"
    elif sc > 0.05:
        sent = "Positive"
        prio = "Normal"
    else:
        sent = "Neutral"
        prio = "Normal"

    # named entities
    doc = ner(eng)
    ents = [f"{e.label_}: {e.text}" for e in doc.ents]
    ent_str = "none" if len(ents) == 0 else "\n".join(ents)

    b_txt = brief(eng, dept, prio, ent_str)
    r_txt = reply(eng, dept, sent, prio, ent_str)

    return scores, f"{sent} ({sc})", prio, ent_str, lang, eng_show, b_txt, r_txt

print("done")

done


In [15]:

print(proc("My Amazon order from last Friday never arrived in Pune and nobody is helping me, worst service!"))
print("=" * 40)
print(proc("माझी डेल्टा एअरलाइन्सची फ्लाइट रद्द झाली आणि माझी बॅग हरवली आहे."))
print("=" * 40)
print(proc("मेरा इंटरनेट कल रात से काम नहीं कर रहा है और कोई मदद नहीं कर रहा।"))

({'Billing': 0.0028096910604267504, 'Retail': 0.9780849691021654, 'Tech Support': 0.001062948486870994, 'Telecom': 0.00774576430521361, 'Travel': 0.010296627045323345}, 'Negative (-0.3595)', 'Medium Priority', 'ORG: Amazon\nDATE: last Friday\nGPE: Pune', 'en', 'not needed', "Summary: My Amazon order from last Friday never arrived in Pune and nobody is helping me, worst service!\n\nNext Steps:\n- Check the status of my order on Amazon's website\n- Contact Amazon's customer support department in Pune\n- Report the issue to the relevant authorities", 'Hello, I am very sorry to hear that your Amazon order from last Friday has not yet arrived in Pune and that you have not received the support you need. We understand the urgency of this situation and are escalating your inquiry to our retail team right away to investigate the delivery status. Please reply to this message with your specific order number so we can look into this matter and provide a proper resolution immediately.')
({'Billing'

In [16]:
import joblib
from google.colab import files

fname = 'nlp_model.pkl'
joblib.dump(clf, fname)

print(f"done {fname}")
files.download(fname)

done nlp_model.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
import re, html, tempfile
from datetime import datetime
import pandas as pd
import gradio as gr

# ---------- config ----------
DEPTS = list(clf.classes_)
PRIOS = ["High Priority", "Medium Priority", "Normal"]
STATUSES = ["New", "In Progress", "Escalated", "Resolved"]
PRIO_RANK = {p: i for i, p in enumerate(PRIOS)}
SLA = {"High Priority": "24 hours", "Medium Priority": "3 days", "Normal": "7 days"}
URGENT_WORDS = ["urgent", "urgently", "emergency", "immediately", "danger", "dangerous", "accident",
                "fire", "flood", "injury", "injured", "unsafe", "hazard", "death", "leak",
                "electric shock", "sewage", "outbreak", "collapse"]
COLS = ["ID", "Logged", "Department", "Priority", "Sentiment", "Language", "Status", "Preview"]
e = html.escape

# ---------- palette ----------
NAVY    = "#14284f"
ROYAL   = "#1f3d85"
BLUE    = "#2f5aa8"
STEEL   = "#4a6aa5"
SKY     = "#3b8cf0"
PINK    = "#ec3fb0"
ROSE    = "#f48ac0"
GREEN   = "#7cc243"
SAGE    = "#5fb36b"
GREY    = "#9aa3b5"
AMBER   = "#f0a030"
PAGE_BG = "#eceef3"
STRIP   = "#f1f1f4"

PRIO_COLOR   = {"High Priority": PINK, "Medium Priority": AMBER, "Normal": SAGE}
SENT_COLOR   = {"Very Negative": PINK, "Negative": ROSE, "Neutral": GREY, "Positive": SAGE}
STATUS_COLOR = {"New": SKY, "In Progress": AMBER, "Escalated": PINK, "Resolved": SAGE}
LEVEL_COLOR  = {"High": SAGE, "Medium": AMBER, "Low": PINK}
DEPT_PALETTE = [PINK, STEEL, SKY, SAGE, GREY, ROYAL, GREEN, "#7a5fc0", "#3aa6b5", ROSE]

def dept_color(d):
    return DEPT_PALETTE[DEPTS.index(d) % len(DEPT_PALETTE)] if d in DEPTS else GREY

CSS = f"""
body, .gradio-container, .gradio-container[class*="gradio-container"] {{
    background: {PAGE_BG} !important;
    color: #1e293b;
    max-width: 100% !important;
    width: 100% !important;
    margin: 0 auto !important;
    padding: 8px 18px !important;
}}
input[type=radio], input[type=checkbox] {{accent-color: {PINK};}}
textarea:focus, input:focus {{border-color: {BLUE} !important; box-shadow: 0 0 0 2px {BLUE}33 !important;}}
button[role="tab"] {{font-weight: 600;}}
button[role="tab"][aria-selected="true"] {{color: {ROYAL} !important; border-bottom: 3px solid {PINK} !important;}}

/* header */
.hero {{background: linear-gradient(135deg,{NAVY},#1b3670 60%,{BLUE}); color: #fff; border-radius: 12px;
       padding: 12px 20px; margin-bottom: 6px; border-bottom: 4px solid {PINK};}}
.hero h2, .hero h2 span {{margin: 0; font-size: 21px; color: #ffffff !important; font-weight: 700; letter-spacing: .02em;}}
.hero h2 span {{font-weight: 300 !important; opacity: 0.92;}}
.hero p {{margin: 3px 0 0; opacity: .88; font-size: 13px; color: #dbe4f5 !important;}}

/* section headings */
.sec {{font-weight: 700; font-size: 14px; color: {NAVY}; margin: 6px 0 6px; padding-bottom: 4px;
      border-bottom: 3px solid transparent; border-image: linear-gradient(90deg,{PINK},{BLUE}) 1;}}

/* KPI cards */
.cards {{display: grid; grid-template-columns: repeat(4, minmax(0, 1fr)); gap: 10px;}}
.card {{background: #fff; border: 1px solid #e3e6ee; border-top: 4px solid {BLUE}; border-radius: 10px;
       overflow: hidden; min-width: 0; transition: transform .15s, box-shadow .15s;}}
.card:hover {{transform: translateY(-2px); box-shadow: 0 6px 14px rgba(20,40,79,.12);}}
.card.placeholder {{opacity: 0.72; border-top-color: #cbd2e0;}}
.card .head {{padding: 9px 12px 7px;}}
.card .lbl {{font-size: 11px; text-transform: uppercase; letter-spacing: .05em; font-weight: 600;
            white-space: nowrap; overflow: hidden; text-overflow: ellipsis;}}
.card .val {{font-size: 19px; font-weight: 700; margin-top: 4px; color: {ROYAL}; word-break: break-word; line-height: 1.2;}}
.card.placeholder .val {{color: #94a3b8;}}
.card .sub {{background: {STRIP}; padding: 5px 12px; font-size: 11.5px; color: #475569; min-height: 16px;
            white-space: nowrap; overflow: hidden; text-overflow: ellipsis;}}

/* badges and chips */
.badge {{display: inline-block; padding: 2px 11px; border-radius: 999px; font-size: 13.5px; font-weight: 700;}}
.chip {{display: inline-block; padding: 2px 10px; border-radius: 999px; font-size: 13px; font-weight: 600; margin-right: 6px;}}

/* banners */
.empty {{padding: 20px; text-align: center; color: #64748b; background: #fff; border: 1px dashed #cbd2e0; border-radius: 10px; font-size: 13.5px;}}
.err-box {{padding: 12px 14px; border-radius: 10px; background: {PINK}1a; color: #b3157f; border-left: 5px solid {PINK};}}
.warn-box {{padding: 8px 12px; border-radius: 10px; background: {AMBER}22; color: #9a6410;
           border-left: 5px solid {AMBER}; margin-bottom: 10px; font-size: 13px;}}

/* department score bars */
.bars {{background: #fff; border: 1px solid #e3e6ee; border-radius: 10px; padding: 6px 14px;}}
.bar-row {{display: grid; grid-template-columns: minmax(105px, 150px) 1fr 44px; gap: 10px; align-items: center; margin: 7px 0;}}
.bar-name {{font-weight: 600; font-size: 13.5px; color: {NAVY};}}
.bar-track {{background: #e5e8f0; border-radius: 999px; height: 11px; overflow: hidden;}}
.bar-fill {{height: 100%; border-radius: 999px; animation: grow .7s ease;}}
.bar-pct {{text-align: right; font-weight: 700; font-size: 13.5px; color: {ROYAL};}}
.sel-tag {{font-size: 10px; background: {PINK}; color: #fff; padding: 1px 7px; border-radius: 999px;
          margin-left: 5px; vertical-align: middle;}}
@keyframes grow {{from {{width: 0;}}}}

/* detail panel */
.detail {{background: #fff; border: 1px solid #e3e6ee; border-top: 4px solid var(--c);
         border-radius: 10px; padding: 12px 14px;}}
.quote {{border-left: 4px solid var(--c); padding: 8px 12px; margin: 8px 0; background: {STRIP};
        border-radius: 0 8px 8px 0;}}

/* buttons */
button.btn-analyze {{background: linear-gradient(135deg,{BLUE},{SKY}) !important; color: #fff !important; border: none !important;}}
button.btn-log     {{background: linear-gradient(135deg,#d62a98,{PINK}) !important; color: #fff !important; border: none !important;}}
button.btn-export  {{background: linear-gradient(135deg,{NAVY},{BLUE}) !important; color: #fff !important; border: none !important;}}
button.btn-update  {{background: linear-gradient(135deg,{ROYAL},{STEEL}) !important; color: #fff !important; border: none !important;}}
button.btn-clear   {{background: #e1e5ee !important; color: #334155 !important; border: none !important;}}
button.btn-analyze, button.btn-log, button.btn-export, button.btn-update, button.btn-clear
  {{transition: transform .12s, filter .12s; font-weight: 600 !important;}}
button.btn-analyze:hover, button.btn-log:hover, button.btn-export:hover,
button.btn-update:hover, button.btn-clear:hover {{filter: brightness(1.08); transform: translateY(-1px);}}

/* text areas */
.box-brief textarea {{border-left: 5px solid {BLUE} !important;}}
.box-reply textarea {{border-left: 5px solid {PINK} !important;}}

@media (max-width: 1100px) {{.cards {{grid-template-columns: repeat(2, minmax(0, 1fr));}}}}
@media (max-width: 600px)  {{.cards {{grid-template-columns: 1fr;}}
                            .bar-row {{grid-template-columns: 100px 1fr 40px;}}}}
"""

def sec(title):
    return f'<div class="sec">{title}</div>'

def chip(text, c):
    return f'<span class="chip" style="background:{c}22;color:{c}">{e(str(text))}</span>'

# ---------- rendering helpers ----------
def card(label, value_html, sub="", color=BLUE, extra_cls=""):
    sub_html = f'<div class="sub" title="{e(sub)}">{e(sub)}</div>'
    return (f'<div class="card {extra_cls}" style="border-top-color:{color}">'
            f'<div class="head"><div class="lbl" style="color:{color}">{label}</div>'
            f'<div class="val">{value_html}</div></div>{sub_html}</div>')

EMPTY_CARDS = '<div class="cards">' + "".join([
    card("Complaint ID", "—", "Awaiting ticket", "#94a3b8", "placeholder"),
    card("Department", "—", "Not routed yet", "#94a3b8", "placeholder"),
    card("Priority", "—", "SLA pending", "#94a3b8", "placeholder"),
    card("Sentiment", "—", "Polarity score", "#94a3b8", "placeholder"),
    card("Language", "—", "EN / HI / MR", "#94a3b8", "placeholder"),
    card("Routing confidence", "—", "Runner-up pending", "#94a3b8", "placeholder"),
    card("Entities found", "—", "NER tags", "#94a3b8", "placeholder"),
    card("Urgency signals", "—", "Keyword scan", "#94a3b8", "placeholder"),
]) + '</div>'

EMPTY_BARS = '<div class="empty">Department confidence scores will appear here after you analyze a complaint.</div>'
EMPTY_DETAIL = '<div class="empty">Click a row in the table to see its details.</div>'

def build_summary(cur):
    ranked = sorted(cur["scores"].items(), key=lambda kv: kv[1], reverse=True)
    top_d, top_p = ranked[0]
    run_d, run_p = ranked[1] if len(ranked) > 1 else ("-", 0.0)
    level = "High" if top_p >= 0.70 else "Medium" if top_p >= 0.45 else "Low"

    dept_overridden = cur["dept"] != cur["pred_dept"]
    prio_overridden = cur["prio"] != cur["pred_prio"]
    dept_sub = (f"Overridden (model said {cur['pred_dept']})" if dept_overridden
                else f"{top_p*100:.0f}% model confidence")
    prio_sub = f"Respond within {SLA.get(cur['prio'], '-')}" + (" · overridden" if prio_overridden else "")

    pc = PRIO_COLOR.get(cur["prio"], SAGE)
    badge = f'<span class="badge" style="background:{pc}22;color:{pc}">{e(cur["prio"])}</span>'

    ent_lines = [] if cur["ents"] == "none" else cur["ents"].split("\n")
    ent_types = sorted({l.split(":")[0] for l in ent_lines})
    words = len(cur["eng"].split())
    urgent = [w for w in URGENT_WORDS if re.search(rf"\b{re.escape(w)}\b", cur["eng"].lower())]

    banner = ""
    if level == "Low" and not dept_overridden:
        banner = ('<div class="warn-box"><b>Low routing confidence.</b> The model is unsure about the '
                  'department. Please verify or override it before routing.</div>')

    cards = "".join([
        card("Complaint ID", e(cur["id"]), cur["time"], NAVY),
        card("Department", e(cur["dept"]), dept_sub, dept_color(cur["dept"])),
        card("Priority", badge, prio_sub, pc),
        card("Sentiment", e(cur["sent_label"]), f"score {cur['score']:+.2f}",
             SENT_COLOR.get(cur["sent_label"], GREY)),
        card("Language", e(cur["lang"].upper()),
             f"{words} words · " + ("original in English" if cur["lang"] == "en" else "translated to English"),
             BLUE),
        card("Routing confidence", level, f"Runner-up: {run_d} ({run_p*100:.0f}%)", LEVEL_COLOR[level]),
        card("Entities found", str(len(ent_lines)), ", ".join(ent_types) if ent_types else "none", SKY),
        card("Urgency signals", str(len(urgent)), ", ".join(urgent) if urgent else "none detected",
             PINK if urgent else GREEN),
    ])
    return banner + f'<div class="cards">{cards}</div>'

def build_bars(scores, selected=None):
    if not scores:
        return EMPTY_BARS
    rows = ""
    for d, p in sorted(scores.items(), key=lambda kv: kv[1], reverse=True):
        c, w = dept_color(d), p * 100
        tag = '<span class="sel-tag">selected</span>' if d == selected else ""
        rows += (f'<div class="bar-row"><div class="bar-name">{e(d)}{tag}</div>'
                 f'<div class="bar-track"><div class="bar-fill" style="width:{w:.1f}%;'
                 f'background:linear-gradient(90deg,{c}99,{c})"></div></div>'
                 f'<div class="bar-pct">{w:.0f}%</div></div>')
    return f'<div class="bars">{rows}</div>'

def style_df(df):
    """Softly color the Department / Priority / Sentiment / Status cells of the log table."""
    if df.empty:
        return df
    def row_style(row):
        out = []
        for col in df.columns:
            v = row[col]
            c = (PRIO_COLOR.get(v) if col == "Priority" else
                 STATUS_COLOR.get(v) if col == "Status" else
                 SENT_COLOR.get(v) if col == "Sentiment" else
                 dept_color(v) if col == "Department" else None)
            out.append(f"background-color:{c}26;color:#1e293b;font-weight:600" if c else "")
        return out
    return df.style.apply(row_style, axis=1)

# ---------- analyze tab logic ----------
def analyze(txt):
    scores, sent_str, prio, ents, lang, eng_show, b, r = proc(txt)
    if scores is None and sent_str == "":
        return EMPTY_CARDS, EMPTY_BARS, "", "", "", "", None, None, None, gr.update(visible=False)
    if scores is None:
        err = f'<div class="err-box"><b>{e(str(b))}</b><br>{e(str(r))}</div>'
        return err, EMPTY_BARS, "", eng_show, "", "", None, None, None, gr.update(visible=False)

    m = re.match(r"(.+?) \((-?[\d.]+)\)\s*$", sent_str)
    sent_label = m.group(1) if m else sent_str
    score = float(m.group(2)) if m else 0.0
    eng = txt if lang == "en" else eng_show
    top = max(scores, key=scores.get)

    now = datetime.now()
    cur = {
        "id": "CMP-" + now.strftime("%y%m%d-%H%M%S"), "time": now.strftime("%d %b %Y, %H:%M"),
        "text": txt, "eng": eng, "lang": lang, "scores": scores,
        "pred_dept": top, "dept": top, "pred_prio": prio, "prio": prio,
        "sent_label": sent_label, "score": score, "ents": ents,
    }
    return build_summary(cur), build_bars(scores, top), ents, eng_show, b, r, top, prio, cur, gr.update(visible=True)

def apply_override(dept, prio, cur):
    """Officer changed department or priority: rebuild cards, bars, brief and reply."""
    if not cur:
        return EMPTY_CARDS, "", "", None, EMPTY_BARS
    cur = dict(cur)
    cur["dept"], cur["prio"] = dept or cur["dept"], prio or cur["prio"]
    b = brief(cur["eng"], cur["dept"], cur["prio"], cur["ents"])
    r = reply(cur["eng"], cur["dept"], cur["sent_label"], cur["prio"], cur["ents"])
    return build_summary(cur), b, r, cur, build_bars(cur["scores"], cur["dept"])

def log_complaint(cur, brief_txt, reply_txt, history):
    history = list(history or [])
    if not cur:
        gr.Warning("Analyze a complaint first.")
        return history
    if any(h["id"] == cur["id"] for h in history):
        gr.Info("This complaint is already logged.")
        return history
    history.append({
        "id": cur["id"], "time": cur["time"], "dept": cur["dept"], "prio": cur["prio"],
        "sent": cur["sent_label"], "lang": cur["lang"], "status": "New",
        "text": cur["text"], "eng": cur["eng"], "brief": brief_txt, "reply": reply_txt,
        "overridden": cur["dept"] != cur["pred_dept"] or cur["prio"] != cur["pred_prio"],
    })
    gr.Info(f"Logged {cur['id']} → {cur['dept']}")
    return history

def clear_all():
    return ["", EMPTY_CARDS, EMPTY_BARS, "", "", "", "", None, None, None, gr.update(visible=False)]

# ---------- log tab logic ----------
def refresh(history, search, prios, statuses):
    history = history or []
    q = (search or "").strip().lower()
    rows = []
    for h in reversed(history):
        if h["prio"] not in (prios or []) or h["status"] not in (statuses or []):
            continue
        if q and q not in " ".join([h["id"], h["dept"], h["text"], h["eng"]]).lower():
            continue
        preview = h["text"].replace("\n", " ")
        rows.append([h["id"], h["time"], h["dept"], h["prio"], h["sent"], h["lang"].upper(),
                     h["status"], preview[:70] + ("…" if len(preview) > 70 else "")])
    rows.sort(key=lambda r: PRIO_RANK.get(r[3], 9))
    return style_df(pd.DataFrame(rows, columns=COLS))

def detail_html(history, cid):
    h = next((x for x in (history or []) if x["id"] == cid), None)
    if not h:
        return EMPTY_DETAIL
    pc = PRIO_COLOR.get(h["prio"], SAGE)
    out = (f'<div class="detail" style="--c:{pc}"><div><b>{e(h["id"])}</b> &nbsp;'
           f'{chip(h["dept"], dept_color(h["dept"]))}{chip(h["prio"], pc)}'
           f'{chip(h["status"], STATUS_COLOR.get(h["status"], GREY))}'
           f'{chip(h["sent"], SENT_COLOR.get(h["sent"], GREY))}</div>'
           f'<div class="quote" style="--c:{pc}">{e(h["text"])}</div>')
    if h["lang"] != "en":
        out += f'<div><i>English:</i> {e(h["eng"])}</div>'
    if h["overridden"]:
        out += '<div style="margin-top:8px">' + chip("Routing manually adjusted by an officer", BLUE) + '</div>'
    return out + '</div>'

def on_select(df, history, evt: gr.SelectData):
    row_value = getattr(evt, "row_value", None)
    if row_value:
        cid = row_value[0]
    else:
        idx = evt.index[0] if isinstance(evt.index, (list, tuple)) else evt.index
        cid = df.iloc[idx, 0]
    h = next((x for x in (history or []) if x["id"] == cid), None)
    return cid, detail_html(history, cid), (h["status"] if h else "New")

def update_status(history, cid, new_status):
    history = [dict(h) for h in (history or [])]
    if not cid:
        gr.Warning("Click a complaint row first.")
        return history
    for h in history:
        if h["id"] == cid:
            h["status"] = new_status
    gr.Info(f"{cid} → {new_status}")
    return history

def export_csv(history):
    if not history:
        gr.Warning("Nothing to export yet.")
        return gr.File(visible=False)
    df = pd.DataFrame([{
        "ID": h["id"], "Logged": h["time"], "Department": h["dept"], "Priority": h["prio"],
        "Sentiment": h["sent"], "Language": h["lang"], "Status": h["status"],
        "Complaint": h["text"], "English": h["eng"], "Internal brief": h["brief"],
        "Draft reply": h["reply"], "Manually adjusted": h["overridden"],
    } for h in history])
    path = tempfile.NamedTemporaryFile(delete=False, suffix=".csv", prefix="complaints_log_").name
    df.to_csv(path, index=False, encoding="utf-8-sig")
    return gr.File(value=path, visible=True)

# ---------- UI ----------
theme = gr.themes.Soft(primary_hue="blue", secondary_hue="pink", neutral_hue="slate")

with gr.Blocks(title="Complaint Routing System") as ui:
    current = gr.State(None)
    history = gr.State([])
    sel_id = gr.State(None)

    gr.HTML('<div class="hero"><h2>COMPLAINT ROUTING <span>DASHBOARD</span></h2>'
            '<p>AI-assisted triage for complaint officers · English, Hindi, Marathi</p></div>')

    with gr.Tabs():
        # ===== TAB 1: ANALYZE =====
        with gr.Tab("Analyze"):
            with gr.Row(equal_height=False):
                with gr.Column(scale=4, min_width=310):
                    gr.HTML(sec("Complaint input"))
                    inp = gr.Textbox(lines=5, show_label=False,
                                     placeholder="Paste or type the complaint here...")
                    with gr.Row():
                        clear_btn = gr.Button("Clear", min_width=0, elem_classes="btn-clear")
                        run_btn = gr.Button("Analyze complaint", min_width=0, elem_classes="btn-analyze")

                    with gr.Column(visible=False) as review_box:
                        gr.HTML(sec("Officer review"))
                        dept_dd = gr.Dropdown(DEPTS, label="Department (change to override)")
                        prio_r = gr.Radio(PRIOS, label="Priority (change to override)")
                        log_btn = gr.Button("✓ Confirm & log complaint", elem_classes="btn-log")

                with gr.Column(scale=8):
                    summary_out = gr.HTML(EMPTY_CARDS)
                    with gr.Tabs():
                        with gr.Tab("Department scores"):
                            dept_out = gr.HTML(EMPTY_BARS)
                        with gr.Tab("Entities"):
                            ents_out = gr.Textbox(lines=3, show_label=False, interactive=False)
                        with gr.Tab("Translation"):
                            eng_out = gr.Textbox(lines=3, show_label=False, interactive=False)

            gr.HTML(sec("Action items"))
            with gr.Row():
                brief_out = gr.Textbox(label="Internal brief", lines=4, buttons=["copy"],
                                       elem_classes="box-brief")
                reply_out = gr.Textbox(label="Draft reply to citizen (editable)", lines=4,
                                       buttons=["copy"], elem_classes="box-reply")

        # ===== TAB 2: LOG =====
        with gr.Tab("Complaint log"):
            gr.HTML(sec("Filter complaints"))
            with gr.Row():
                search = gr.Textbox(label="Search", placeholder="ID, department, or complaint text...", scale=3)
                prio_f = gr.CheckboxGroup(PRIOS, value=PRIOS, label="Priority", scale=3)
                status_f = gr.CheckboxGroup(STATUSES, value=STATUSES, label="Status", scale=3)
            table = gr.Dataframe(value=pd.DataFrame(columns=COLS), headers=COLS,
                                 interactive=False, wrap=True, show_label=False)
            gr.HTML(sec("Selected complaint"))
            detail = gr.HTML(EMPTY_DETAIL)
            with gr.Row():
                status_dd = gr.Dropdown(STATUSES, value="New", label="Set status for selected complaint", scale=3)
                upd_btn = gr.Button("Update status", elem_classes="btn-update", scale=1)
                exp_btn = gr.Button("Export CSV", elem_classes="btn-export", scale=1)
            file_out = gr.File(label="Download", visible=False)

    # ---------- wiring ----------
    analyze_outs = [summary_out, dept_out, ents_out, eng_out, brief_out, reply_out, dept_dd, prio_r, current, review_box]
    run_btn.click(analyze, inp, analyze_outs)
    clear_btn.click(clear_all, None, [inp] + analyze_outs)

    override_outs = [summary_out, brief_out, reply_out, current, dept_out]
    dept_dd.input(apply_override, [dept_dd, prio_r, current], override_outs)
    prio_r.input(apply_override, [dept_dd, prio_r, current], override_outs)

    refresh_in, refresh_out = [history, search, prio_f, status_f], [table]
    log_btn.click(log_complaint, [current, brief_out, reply_out, history], history).then(
        refresh, refresh_in, refresh_out)
    for comp in (search, prio_f, status_f):
        comp.change(refresh, refresh_in, refresh_out)

    table.select(on_select, [table, history], [sel_id, detail, status_dd])
    upd_btn.click(update_status, [history, sel_id, status_dd], history).then(
        refresh, refresh_in, refresh_out).then(detail_html, [history, sel_id], detail)
    exp_btn.click(export_csv, history, file_out)

ui.launch(share=True, height=900, theme=theme, css=CSS)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://72ee05830d1229666a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
